# GazeToolkit vs. Tobii Pro Lab — Evaluation

Evaluates [GazeToolkit (Konopka, 2019)](https://github.com/uxifiit/GazeToolkit) against Tobii Pro Lab ground truth on one eye-tracking recording.

## Setup
Both systems implement the **I-VT (Velocity-Threshold) fixation filter** with the same core settings:
- Eye selection: **both-eye average**
- Velocity threshold: **30 °/s**
- Velocity window: **20 ms**

## Label alignment
GazeToolkit uses three output labels: `Fixation`, `Saccade`, `Unknown`.  
Tobii Pro Lab uses four: `Fixation`, `Saccade`, `Unclassified`, `EyesNotFound`.

To ensure a symmetric comparison, the **ground truth is collapsed** to match GazeToolkit's coarser schema:

| Tobii Pro Lab (GT) | → Collapsed GT |
|--------------------|----------------|
| Fixation | Fixation |
| Saccade | Saccade |
| Unclassified | **Unknown** |
| EyesNotFound | **Unknown** |

GazeToolkit's `Unknown` output is kept as-is — no post-hoc patching.

## Two variants
| Variant | GazeToolkit source | GT source |
|---------|-------------------|-----------|
| **No gap fill** | Pre-computed C# pipeline output | No-gap-fill Tobii export |
| **Gap fill 75 ms** | `ivt_filter.py` (Python) on Tobii-gap-filled data | Gap-fill Tobii export |

> **Limitation (gap-fill variant):** `ivt_filter.py` does not implement gap fill itself. It runs on gaze data that Tobii already gap-filled during export. The comparison is therefore *GazeToolkit on Tobii-gap-filled data* vs *Tobii Pro Lab with gap fill 75 ms*, not *GazeToolkit with gap fill*.

---
## 1 — Imports & Paths

In [1]:
import importlib.util
import json
import math
import os
import shutil
import sys
import tempfile
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd

# Make the repo root importable (works whether the notebook is run from
# notebooks/ or from the repo root)
REPO_ROOT = Path("__file__").resolve().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ivt_filter.evaluation.evaluation import compute_ivt_metrics
from ivt_filter.evaluation.event_iou import compute_event_iou_metrics

print("Imports OK")

Imports OK


In [2]:
GAZETOOLKIT_DIR = Path("/home/cem/Documents/Gitprojekt/GazeToolkit")
RESULTS_DIR     = REPO_ROOT / "results" / "gazetoolkit"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# --- no-gap-fill variant ---
NGF_INPUT       = GAZETOOLKIT_DIR / "I-VT-botheye20ms30threshold_input.csv"
NGF_IVT_OUT     = RESULTS_DIR / "ivt_exe_no_gap_fill.csv"   # i-vt.exe, no --fillin

# --- gap-fill variant ---
GF_TOBII_TSV    = (
    GAZETOOLKIT_DIR / "dataset"
    / "IVT-Interpolation75ms-Eyeboth-NoNoise-VelocityWindow20-VelocityTreshold30.tsv"
)
GF_IVT_OUT      = RESULTS_DIR / "ivt_exe_gap_fill.csv"       # i-vt.exe, --fillin 75ms

# --- output files ---
RESULTS_JSON    = RESULTS_DIR / "gazetoolkit_eval_results.json"
SUMMARY_TXT     = RESULTS_DIR / "gazetoolkit_summary.txt"

# Tobii raw-export column -> slim name
TOBII_COL_MAP: Dict[str, str] = {
    "Gaze point left X [DACS mm]"  : "gaze_left_x_mm",
    "Gaze point left Y [DACS mm]"  : "gaze_left_y_mm",
    "Gaze point right X [DACS mm]" : "gaze_right_x_mm",
    "Gaze point right Y [DACS mm]" : "gaze_right_y_mm",
    "Gaze point left X [DACS px]"  : "gaze_left_x_px",
    "Gaze point left Y [DACS px]"  : "gaze_left_y_px",
    "Gaze point right X [DACS px]" : "gaze_right_x_px",
    "Gaze point right Y [DACS px]" : "gaze_right_y_px",
    "Validity left"                : "validity_left",
    "Validity right"               : "validity_right",
    "Eye position left X [DACS mm]": "eye_left_x_mm",
    "Eye position left Y [DACS mm]": "eye_left_y_mm",
    "Eye position left Z [DACS mm]": "eye_left_z_mm",
    "Eye position right X [DACS mm]": "eye_right_x_mm",
    "Eye position right Y [DACS mm]": "eye_right_y_mm",
    "Eye position right Z [DACS mm]": "eye_right_z_mm",
    "Eye movement type"            : "gt_event_type",
    "Eye movement type index"      : "gt_event_index",
}

TIMESTAMP_FALLBACKS = [
    ("Recording timestamp [ms]", 1.0),
    ("Recording timestamp [\u03bcs]", 1e-3),
    ("Eyetracker timestamp [\u03bcs]", 1e-3),
]

print("Paths OK")


Paths OK


---
## 2 — Helper Functions

In [3]:
# ── label schemas ─────────────────────────────────────────────────────────────
GT_LABEL_MAP = {"Fixation": "Fixation", "Saccade": "Saccade", "Unknown": "Unknown"}

def collapse_gt(series: pd.Series) -> pd.Series:
    """Collapse Tobii GT to 3-class schema: EyesNotFound + Unclassified -> Unknown."""
    mapping = {"EyesNotFound": "Unknown", "Unclassified": "Unknown"}
    return series.fillna("Unknown").astype(str).replace(mapping)


# ── data loading ─────────────────────────────────────────────────────────────
def load_gazetoolkit_input(path: Path) -> pd.DataFrame:
    """Load slim GazeToolkit input CSV (auto-detect separator)."""
    df = pd.read_csv(path, sep=None, engine="python")
    assert "time_ms" in df.columns and "gt_event_type" in df.columns
    return df.sort_values("time_ms").reset_index(drop=True)


def load_ivt_exe_output(path: Path) -> pd.DataFrame:
    """
    Load i-vt.exe CSV output.
    Converts ISO 8601 timestamps (Unix epoch + ms) to start_time / end_time in ms.
    """
    df = pd.read_csv(path)
    epoch = pd.Timestamp("1970-01-01", tz="UTC")
    ts = pd.to_datetime(df["Timestamp"], utc=True)
    df["start_time"] = (ts - epoch).dt.total_seconds() * 1000
    df["end_time"]   = df["start_time"] + df["Duration"]
    df = df.rename(columns={"MovementType": "movement_type"})
    return df[["movement_type", "start_time", "end_time"]].sort_values("start_time").reset_index(drop=True)


def extract_tobii_input(tobii_tsv: Path) -> pd.DataFrame:
    """Extract slim input from 92-column Tobii Pro Lab export."""
    df_raw = pd.read_csv(tobii_tsv, sep="\t", decimal=",", low_memory=False)
    ts_col, ts_scale = None, 1.0
    for col, scale in TIMESTAMP_FALLBACKS:
        if col in df_raw.columns:
            ts_col, ts_scale = col, scale
            break
    if ts_col is None:
        raise ValueError("No timestamp column found.")
    print(f"  Timestamp: '{ts_col}' x {ts_scale}")
    available = {k: v for k, v in TOBII_COL_MAP.items() if k in df_raw.columns}
    available[ts_col] = "time_ms"
    df_slim = df_raw[list(available.keys())].rename(columns=available)
    if ts_scale != 1.0:
        df_slim["time_ms"] = df_slim["time_ms"] * ts_scale
    return df_slim.sort_values("time_ms").reset_index(drop=True)


# ── timestamp alignment ───────────────────────────────────────────────────────
def assign_gazetoolkit_labels(samples_df: pd.DataFrame,
                               events_df: pd.DataFrame) -> pd.Series:
    """Map event-level i-vt.exe output to sample-level labels via merge_asof."""
    samples = samples_df[["time_ms"]].copy().sort_values("time_ms")
    samples["time_ms"] = samples["time_ms"].astype("float64")
    events = events_df[["start_time", "end_time", "movement_type"]].sort_values("start_time").reset_index(drop=True)
    merged = pd.merge_asof(samples, events, left_on="time_ms", right_on="start_time", direction="backward")
    not_covered = merged["end_time"].isna() | (merged["time_ms"] > merged["end_time"])
    merged.loc[not_covered, "movement_type"] = "Unknown"
    labels = merged["movement_type"].map(GT_LABEL_MAP).fillna("Unknown")
    labels.index = samples.index
    return labels.reindex(samples_df.index)


print("Helper functions defined")


Helper functions defined


In [4]:
# ── metric helpers ────────────────────────────────────────────────────────────

def _json_safe(val: Any) -> Any:
    if isinstance(val, float) and math.isnan(val):
        return None
    if isinstance(val, (list, tuple)):
        return [_json_safe(v) for v in val]
    if isinstance(val, dict):
        return {k: _json_safe(v) for k, v in val.items()}
    return val


def _prf(tp: int, n_gt: int, n_pred: int) -> Dict[str, Any]:
    rec  = tp / n_gt   if n_gt   > 0 else float("nan")
    prec = tp / n_pred if n_pred > 0 else float("nan")
    denom = rec + prec
    f1 = (2 * prec * rec / denom
          if not math.isnan(prec) and not math.isnan(rec) and denom > 0
          else float("nan"))
    return dict(
        precision = round(prec * 100, 1) if not math.isnan(prec) else None,
        recall    = round(rec  * 100, 1) if not math.isnan(rec)  else None,
        f1        = round(f1   * 100, 1) if not math.isnan(f1)   else None,
    )


def extract_prf_from_confusion(m: Dict[str, Any]) -> Dict[str, Dict]:
    """Compute per-class precision, recall, F1 from compute_ivt_metrics output."""
    labels: List[str] = m["labels"]
    conf:   List[List[int]] = m["confusion_matrix"]
    k = len(labels)
    result = {}
    for cls in ["Fixation", "Saccade"]:
        if cls not in labels:
            result[cls] = dict(precision=None, recall=None, f1=None)
            continue
        idx    = labels.index(cls)
        tp     = conf[idx][idx]
        n_gt   = sum(conf[idx])
        n_pred = sum(conf[i][idx] for i in range(k))
        result[cls] = _prf(tp, n_gt, n_pred)
    return result


def extract_event_prf(m: Dict[str, Any]) -> Dict[str, Dict]:
    """Compute per-class P/R/F1 from compute_event_iou_metrics output."""
    cm     = m.get("confusion_matrix", {})
    counts = m.get("event_counts", {})
    result = {}
    for cls in ["Fixation", "Saccade"]:
        gt_row = cm.get(cls, {})
        tp     = gt_row.get(cls, 0)
        fn     = gt_row.get("FN", 0)
        wrong  = sum(v for tag, v in gt_row.items() if tag not in (cls, "FN"))
        n_gt   = tp + fn + wrong
        n_pred = counts.get(cls, {}).get("pred", 0)
        d = _prf(tp, n_gt, n_pred)
        d["n_gt"] = n_gt
        d["n_pred"] = n_pred
        result[cls] = d
    return result


def display_results(sample_m: Dict, event_m: Dict, variant_name: str) -> None:
    """Pretty-print evaluation results for one variant."""
    prf_s = extract_prf_from_confusion(sample_m)
    prf_e = extract_event_prf(event_m)

    print(f"\n{'='*55}")
    print(f"  {variant_name}")
    print(f"{'='*55}")

    print("\nSample-level")
    print(f"  Agreement (GT Fix/Sac only) : {sample_m['percentage_agreement']:.1f} %")
    print(f"  Agreement (all samples)     : {sample_m['percentage_agreement_all']:.1f} %")
    print(f"  Cohen's κ                   : {sample_m['cohen_kappa']:.3f}")
    print(f"")
    hdr = f"  {'Class':<12} {'Precision':>10} {'Recall':>10} {'F1':>10}"
    print(hdr); print("  " + "-"*42)
    for cls in ["Fixation", "Saccade"]:
        p = prf_s[cls]
        print(f"  {cls:<12} {str(p['precision']):>10} {str(p['recall']):>10} {str(p['f1']):>10}")

    print("\nEvent-level  (max-IoU matching, min IoU > 0)")
    hdr2 = f"  {'Class':<12} {'Precision':>10} {'Recall':>10} {'F1':>10}  {'GT ev':>6}  {'Pred ev':>7}"
    print(hdr2); print("  " + "-"*56)
    for cls in ["Fixation", "Saccade"]:
        p = prf_e[cls]
        print(f"  {cls:<12} {str(p['precision']):>10} {str(p['recall']):>10} "
              f"{str(p['f1']):>10}  {p['n_gt']:>6}  {p['n_pred']:>7}")


print("Metric helpers defined")

Metric helpers defined


In [5]:
# i-vt.exe outputs are pre-computed via the C# pipeline:
#   Variant 1: mono i-vt.exe ... --select Average --window-side 10 --threshold 30
#   Variant 2: mono i-vt.exe ... --fillin --fillin-max-gap 75
# See: tools/convert_tobii_to_gazedata_csv.py for the Tobii->GazeData conversion.
print("C# pipeline outputs ready")


C# pipeline outputs ready


---
## 3 — Variant 1: No Gap Fill

Uses the **pre-computed C# pipeline output** from GazeToolkit.  
Ground truth from the matching Tobii Pro Lab export (no gap fill).

In [6]:
# Load input (ground truth + timestamps)
ngf_samples = load_gazetoolkit_input(NGF_INPUT)
print(f"Samples : {len(ngf_samples):,}  "
      f"({ngf_samples['time_ms'].min():.0f} – {ngf_samples['time_ms'].max():.0f} ms)")
print("GT label distribution (original):")
print(ngf_samples["gt_event_type"].value_counts().to_string())

Samples : 27,801  (114 – 231782 ms)
GT label distribution (original):
gt_event_type
Fixation        20174
EyesNotFound     4132
Saccade          2805
Unclassified      690


In [7]:
# Load i-vt.exe output (no gap fill)
ngf_events = load_ivt_exe_output(NGF_IVT_OUT)

print(f"Events  : {len(ngf_events):,}")
print("Event type distribution:")
print(ngf_events["movement_type"].value_counts().to_string())


Events  : 2,532
Event type distribution:
movement_type
Fixation    1266
Saccade      779
Unknown      487


In [8]:
# Collapse GT labels + assign GazeToolkit predictions
ngf_eval = ngf_samples.copy()
ngf_eval["gt_collapsed"]       = collapse_gt(ngf_eval["gt_event_type"])
ngf_eval["gazetoolkit_label"]  = assign_gazetoolkit_labels(ngf_eval, ngf_events)

print("Collapsed GT distribution:")
print(ngf_eval["gt_collapsed"].value_counts().to_string())
print("\nGazeToolkit prediction distribution:")
print(ngf_eval["gazetoolkit_label"].value_counts().to_string())

Collapsed GT distribution:
gt_collapsed
Fixation    20174
Unknown      4822
Saccade      2805

GazeToolkit prediction distribution:
gazetoolkit_label
Fixation    20891
Unknown      4135
Saccade      2775


In [9]:
# Cross-tabulation (sanity check)
pd.crosstab(
    ngf_eval["gt_collapsed"],
    ngf_eval["gazetoolkit_label"],
    margins=True,
)

gazetoolkit_label,Fixation,Saccade,Unknown,All
gt_collapsed,,,,
Fixation,20074,98,2,20174
Saccade,127,2677,1,2805
Unknown,690,0,4132,4822
All,20891,2775,4135,27801


In [10]:
# Sample-level metrics
ngf_sample_m = compute_ivt_metrics(
    ngf_eval,
    gt_col="gt_collapsed",
    pred_col="gazetoolkit_label",
)

# Event-level metrics (max-IoU)
ngf_event_m = compute_event_iou_metrics(
    ngf_eval,
    gt_col="gt_collapsed",
    pred_col="gazetoolkit_label",
    time_col="time_ms",
    event_types=["Fixation", "Saccade"],
)

display_results(ngf_sample_m, ngf_event_m, "Variant 1 — No Gap Fill  (C# pipeline)")


  Variant 1 — No Gap Fill  (C# pipeline)

Sample-level
  Agreement (GT Fix/Sac only) : 99.0 %
  Agreement (all samples)     : 96.7 %
  Cohen's κ                   : 0.921

  Class         Precision     Recall         F1
  ------------------------------------------
  Fixation           96.1       99.5       97.8
  Saccade            96.5       95.4       95.9

Event-level  (max-IoU matching, min IoU > 0)
  Class         Precision     Recall         F1   GT ev  Pred ev
  --------------------------------------------------------
  Fixation           57.5       98.1       72.5     738     1260
  Saccade            88.6       94.0       91.2     734      779


### Notes — No Gap Fill

- Pipeline run: `i-vt.exe --select Average --frequency 120 --window-side 10 --threshold 30` (no merge, no denoise, no discard).
- **High sample-level F1** for both classes, but **low event-level Fixation precision**: i-vt.exe produces 1260 fixation events vs 738 in the GT. This is expected — without a merge step, adjacent fixations separated by brief saccades are not joined. The sample-level label is still mostly correct (each fragment is still labelled Fixation), hence the disconnect between sample and event agreement.
- The merge step (`--merge --merge-max-gap 75 --merge-max-angle 0.5`) is deliberately omitted to evaluate the pure I-VT classifier.


---
## 4 - Variant 2: Gap Fill 75 ms

Runs **GazeToolkit's C# pipeline** (`i-vt.exe`) on the raw no-gap-fill data **with** `--fillin --fillin-max-gap 75`.
GazeToolkit applies its own linear interpolation before classification.

Ground truth: Tobii Pro Lab export **with** gap fill 75ms enabled.

> Note: Both systems independently apply gap fill to the same raw signal. This is a proper like-for-like comparison.


In [11]:
# GT comes from the gap-fill Tobii export (Tobii applied its own gap fill)
# Sample timestamps match the no-gap-fill recording (same session, same timestamps)
print(f"Loading GT: {GF_TOBII_TSV.name}")
gf_samples = extract_tobii_input(GF_TOBII_TSV)
print(f"Samples : {len(gf_samples):,}  "
      f"({gf_samples['time_ms'].min():.0f} - {gf_samples['time_ms'].max():.0f} ms)")
print("GT label distribution (original):")
print(gf_samples["gt_event_type"].value_counts().to_string())


Loading GT: IVT-Interpolation75ms-Eyeboth-NoNoise-VelocityWindow20-VelocityTreshold30.tsv


  Timestamp: 'Recording timestamp [μs]' x 0.001
Samples : 27,822  (0 - 231939 ms)
GT label distribution (original):
gt_event_type
Fixation        20991
EyesNotFound     3600
Saccade          3226
Unclassified        3


In [12]:
# Load i-vt.exe output (with gap fill 75ms)
gf_events = load_ivt_exe_output(GF_IVT_OUT)

print(f"Events  : {len(gf_events):,}")
print("Event type distribution:")
print(gf_events["movement_type"].value_counts().to_string())


Events  : 1,668
Event type distribution:
movement_type
Fixation    834
Saccade     809
Unknown      25


In [13]:
# Collapse GT labels + assign GazeToolkit predictions
gf_eval = gf_samples.copy()
gf_eval["gt_collapsed"]      = collapse_gt(gf_eval["gt_event_type"])
gf_eval["gazetoolkit_label"] = assign_gazetoolkit_labels(gf_eval, gf_events)

print("Collapsed GT distribution:")
print(gf_eval["gt_collapsed"].value_counts().to_string())
print("\nGazeToolkit prediction distribution:")
print(gf_eval["gazetoolkit_label"].value_counts().to_string())

Collapsed GT distribution:
gt_collapsed
Fixation    20991
Unknown      3605
Saccade      3226

GazeToolkit prediction distribution:
gazetoolkit_label
Fixation    21121
Unknown      3595
Saccade      3106


In [14]:
# Cross-tabulation
pd.crosstab(
    gf_eval["gt_collapsed"],
    gf_eval["gazetoolkit_label"],
    margins=True,
)

gazetoolkit_label,Fixation,Saccade,Unknown,All
gt_collapsed,,,,
Fixation,20923,68,0,20991
Saccade,186,3038,2,3226
Unknown,12,0,3593,3605
All,21121,3106,3595,27822


In [15]:
# Metrics
gf_sample_m = compute_ivt_metrics(
    gf_eval,
    gt_col="gt_collapsed",
    pred_col="gazetoolkit_label",
)

gf_event_m = compute_event_iou_metrics(
    gf_eval,
    gt_col="gt_collapsed",
    pred_col="gazetoolkit_label",
    time_col="time_ms",
    event_types=["Fixation", "Saccade"],
)

display_results(gf_sample_m, gf_event_m, "Variant 2 — Gap Fill 75 ms  (ivt_filter.py)")


  Variant 2 — Gap Fill 75 ms  (ivt_filter.py)

Sample-level
  Agreement (GT Fix/Sac only) : 98.9 %
  Agreement (all samples)     : 99.0 %
  Cohen's κ                   : 0.976

  Class         Precision     Recall         F1
  ------------------------------------------
  Fixation           99.1       99.7       99.4
  Saccade            97.8       94.2       96.0

Event-level  (max-IoU matching, min IoU > 0)
  Class         Precision     Recall         F1   GT ev  Pred ev
  --------------------------------------------------------
  Fixation           92.0       89.5       90.7     857      834
  Saccade            95.3       89.0       92.1     866      809


### Notes - Gap Fill 75 ms

- GazeToolkit applies **linear interpolation** (FillInGapsInterpolation) to the raw signal before classification.
- Both GazeToolkit and Tobii Pro Lab independently fill the same gaps; this is a true like-for-like comparison.
- Unknown events drop from 487 (no gap fill) to 25 (gap fill), confirming successful interpolation.
- Tobii Pro Lab GT (gap-fill export) uses its own classification on the interpolated data.


---
## 5 — Side-by-Side Comparison

In [16]:
def make_summary_df(sample_m, event_m, name):
    prf_s = extract_prf_from_confusion(sample_m)
    prf_e = extract_event_prf(event_m)
    rows = {
        "Variant": name,
        "Agreement Fix/Sac (%)": round(sample_m["percentage_agreement"], 1),
        "Agreement all (%)": round(sample_m["percentage_agreement_all"], 1),
        "Cohen's κ": round(sample_m["cohen_kappa"], 3),
        "Fix Precision (%)": prf_s["Fixation"]["precision"],
        "Fix Recall (%)": prf_s["Fixation"]["recall"],
        "Fix F1 (%)": prf_s["Fixation"]["f1"],
        "Sac Precision (%)": prf_s["Saccade"]["precision"],
        "Sac Recall (%)": prf_s["Saccade"]["recall"],
        "Sac F1 (%)": prf_s["Saccade"]["f1"],
        "Fix F1 event (%)": prf_e["Fixation"]["f1"],
        "Sac F1 event (%)": prf_e["Saccade"]["f1"],
        "GT Fix events": prf_e["Fixation"]["n_gt"],
        "Pred Fix events": prf_e["Fixation"]["n_pred"],
        "GT Sac events": prf_e["Saccade"]["n_gt"],
        "Pred Sac events": prf_e["Saccade"]["n_pred"],
    }
    return rows


comparison = pd.DataFrame([
    make_summary_df(ngf_sample_m, ngf_event_m, "No Gap Fill (C#)"),
    make_summary_df(gf_sample_m,  gf_event_m,  "Gap Fill 75ms (Python)"),
]).set_index("Variant")

comparison

,Agreement Fix/Sac (%),Agreement all (%),Cohen's κ,Fix Precision (%),Fix Recall (%),Fix F1 (%),Sac Precision (%),Sac Recall (%),Sac F1 (%),Fix F1 event (%),Sac F1 event (%),GT Fix events,Pred Fix events,GT Sac events,Pred Sac events
Variant,,,,,,,,,,,,,,,
No Gap Fill (C#),99.0,96.7,0.921,96.1,99.5,97.8,96.5,95.4,95.9,72.5,91.2,738,1260,734,779
Gap Fill 75ms (Python),98.9,99.0,0.976,99.1,99.7,99.4,97.8,94.2,96.0,90.7,92.1,857,834,866,809


---
## 6 — Save Outputs

In [17]:
def serialise_metrics(sample_m, event_m):
    """Make both metric dicts JSON-safe (drop MatchResult lists, handle NaN)."""
    prf_s = extract_prf_from_confusion(sample_m)
    prf_e = extract_event_prf(event_m)
    sl = _json_safe({k: v for k, v in sample_m.items() if k != "confusion_matrix"})
    sl["confusion_matrix"] = _json_safe(sample_m["confusion_matrix"])
    sl["confusion_matrix_labels"] = sample_m["labels"]
    sl["per_class"] = prf_s
    el = _json_safe({k: v for k, v in event_m.items()
                     if k not in ("matches", "unmatched_pred")})
    el["per_class"] = prf_e
    return {"sample_level": sl, "event_level": el}


results = {
    "no_gap_fill": serialise_metrics(ngf_sample_m, ngf_event_m),
    "gap_fill":    serialise_metrics(gf_sample_m,  gf_event_m),
    "label_schema": {
        "gt_collapse": {"EyesNotFound": "Unknown", "Unclassified": "Unknown"},
        "classes": ["Fixation", "Saccade", "Unknown"],
        "pipeline": "GazeToolkit i-vt.exe (C# original)",
    },
}

with open(RESULTS_JSON, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"JSON saved -> {RESULTS_JSON}")


JSON saved -> /home/cem/Documents/Gitprojekt/Tobii-I-VT-Filter-Reconstruction/results/gazetoolkit/gazetoolkit_eval_results.json


In [18]:
# Plain-text summary for thesis
def _f(v, d=1):
    return "N/A" if v is None else f"{v:.{d}f}"

lines = [
    "GazeToolkit vs Tobii Pro Lab — Evaluation Summary",
    "=" * 55,
    "",
    "Settings : both-eye average | threshold 30 °/s | window 20 ms",
    "Labels   : Fixation / Saccade / Unknown",
    "           (Tobii EyesNotFound + Unclassified collapsed → Unknown)",
    "",
]

for variant_key, sm, em, title in [
    ("no_gap_fill", ngf_sample_m, ngf_event_m, "Variant 1 — No Gap Fill  (C# pipeline)"),
    ("gap_fill",    gf_sample_m,  gf_event_m,  "Variant 2 — Gap Fill 75 ms  (ivt_filter.py)"),
]:
    prf_s = extract_prf_from_confusion(sm)
    prf_e = extract_event_prf(em)
    lines += [
        title, "-" * len(title), "",
        "Sample-level",
        f"  Agreement (Fix/Sac only) : {_f(sm['percentage_agreement'])} %",
        f"  Agreement (all samples)  : {_f(sm['percentage_agreement_all'])} %",
        f"  Cohen's κ                : {_f(sm['cohen_kappa'], 3)}",
        f"",
        f"  {'Class':<12} {'Precision':>10} {'Recall':>10} {'F1':>10}",
        f"  {'-'*12} {'-'*10} {'-'*10} {'-'*10}",
    ]
    for cls in ["Fixation", "Saccade"]:
        p = prf_s[cls]
        lines.append(f"  {cls:<12} {_f(p['precision']):>10} "
                     f"{_f(p['recall']):>10} {_f(p['f1']):>10}")
    lines += [
        "",
        "Event-level  (max-IoU, min IoU > 0)",
        f"  {'Class':<12} {'Precision':>10} {'Recall':>10} {'F1':>10}  GT ev  Pred ev",
        f"  {'-'*12} {'-'*10} {'-'*10} {'-'*10}  -----  -------",
    ]
    for cls in ["Fixation", "Saccade"]:
        p = prf_e[cls]
        lines.append(f"  {cls:<12} {_f(p['precision']):>10} "
                     f"{_f(p['recall']):>10} {_f(p['f1']):>10}  "
                     f"{p['n_gt']:>5}  {p['n_pred']:>7}")
    lines.append("")

lines += [
    "Pipeline (both variants): i-vt.exe (GazeToolkit C#, UXI.GazeToolkit.dll)",
    "    Settings: --select Average --frequency 120 --window-side 10 --threshold 30",
    "    No merge, denoise, or discard step applied.",
    "",
    "Limitations",
    "-----------",
    "No gap fill: Fixation over-fragmentation due to missing merge step (intentional,",
    "             to isolate the pure I-VT filter without post-processing).",
    "Gap fill   : GazeToolkit and Tobii Pro Lab apply independent linear interpolation",
    "             to the same raw signal; interpolated values may differ slightly.",
]

summary_text = "\n".join(lines)

with open(SUMMARY_TXT, "w", encoding="utf-8") as f:
    f.write(summary_text)

print(summary_text)

GazeToolkit vs Tobii Pro Lab — Evaluation Summary

Settings : both-eye average | threshold 30 °/s | window 20 ms
Labels   : Fixation / Saccade / Unknown
           (Tobii EyesNotFound + Unclassified collapsed → Unknown)

Variant 1 — No Gap Fill  (C# pipeline)
--------------------------------------

Sample-level
  Agreement (Fix/Sac only) : 99.0 %
  Agreement (all samples)  : 96.7 %
  Cohen's κ                : 0.921

  Class         Precision     Recall         F1
  ------------ ---------- ---------- ----------
  Fixation           96.1       99.5       97.8
  Saccade            96.5       95.4       95.9

Event-level  (max-IoU, min IoU > 0)
  Class         Precision     Recall         F1  GT ev  Pred ev
  ------------ ---------- ---------- ----------  -----  -------
  Fixation           57.5       98.1       72.5    738     1260
  Saccade            88.6       94.0       91.2    734      779

Variant 2 — Gap Fill 75 ms  (ivt_filter.py)
-------------------------------------------

Samp